# 06 · Tools

## What is a tool?

From the [tools docs](https://docs.langchain.com/oss/python/langchain/tools):

> **Tools** are "callable functions with well-defined inputs and outputs that get
> passed to a chat model. The model decides when to invoke a tool based on the
> conversation context, and what input arguments to provide."

Two audiences, one object: **your code** calls it like a function, and **the model**
reads its name, description and argument schema to decide whether to call it.

In [1]:
from typing import Annotated, Literal

from langchain.tools import tool
from langchain_core.messages import AIMessage
from langgraph.graph import END, START, StateGraph, add_messages
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field, ValidationError
from typing_extensions import TypedDict

## Defining a tool

Decorate a function with `@tool`. The docs are strict about two requirements:

> "Type hints are required as they define the tool's input schema. The docstring should
> be informative and concise to help the model understand the tool's purpose."

The type hints *become* the schema and the docstring *becomes* the description.

In [2]:
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b


@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b


@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

## What the decorator produced

`add` is no longer a plain function. It is a tool object carrying the schema derived
from the signature and docstring.

In [3]:
print("name:       ", add.name)
print("description:", add.description)
print("args:       ", add.args)

name:        add
description: Add two numbers together.
args:        {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}


That schema is exactly what gets sent to a model in
[07-models.ipynb](./07-models.ipynb) — nothing is added later. A vague docstring is a
real bug: it is the only thing telling the model what the tool is for.

## Calling it yourself

Use `.invoke()` with a dict of arguments. This is the whole tool-execution story with
no model anywhere near it.

In [4]:
print(add.invoke({"a": 2, "b": 3}))
print(subtract.invoke({"a": 10, "b": 4}))
print(multiply.invoke({"a": 6, "b": 7}))

5.0
6.0
42.0


Arguments are validated against the schema before your function body runs, so a bad
call fails at the boundary rather than inside your code:

In [5]:
try:
    add.invoke({"a": "not a number", "b": 3})
except ValidationError as e:
    print("ValidationError:", e.errors()[0]["msg"])

ValidationError: Input should be a valid number, unable to parse string as a number


## Customising name and description

You can
[override the generated metadata](https://docs.langchain.com/oss/python/langchain/tools#customize-tool-properties)
when the Python name is not the name you want the model to see.

In [6]:
@tool("calculator", description="Evaluate one arithmetic operation on two numbers.")
def calculate(a: float, b: float, op: Literal["add", "sub", "mul"]) -> float:
    """This docstring is ignored — the description= argument wins."""
    return {"add": a + b, "sub": a - b, "mul": a * b}[op]


print("name:       ", calculate.name)
print("description:", calculate.description)
print("op accepts: ", calculate.args["op"])
print("result:     ", calculate.invoke({"a": 6, "b": 7, "op": "mul"}))

name:        calculator
description: Evaluate one arithmetic operation on two numbers.
op accepts:  {'enum': ['add', 'sub', 'mul'], 'title': 'Op', 'type': 'string'}
result:      42.0


`Literal` became an enum in the schema, so a model cannot invent a fourth operation.
Prefer `Literal` over a free-form `str` whenever the valid values are known.

## Richer schemas with Pydantic

For per-argument descriptions and defaults, pass an
[`args_schema`](https://docs.langchain.com/oss/python/langchain/tools#advanced-schema-definition).

In [7]:
class RoundArgs(BaseModel):
    value: float = Field(description="The number to round.")
    places: int = Field(default=2, description="How many decimal places to keep.")


@tool(args_schema=RoundArgs)
def round_to(value: float, places: int = 2) -> float:
    """Round a number to a given number of decimal places."""
    return round(value, places)


for name, spec in round_to.args.items():
    print(f"{name}: {spec}")

print()
print(round_to.invoke({"value": 3.14159}))

value: {'description': 'The number to round.', 'title': 'Value', 'type': 'number'}
places: {'default': 2, 'description': 'How many decimal places to keep.', 'title': 'Places', 'type': 'integer'}

3.14


Those per-argument descriptions travel to the model too, and `places` did not have to
be supplied because the schema carries the default.

## When a tool fails

A tool called directly raises normally — it is still Python.

In [8]:
@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b


try:
    divide.invoke({"a": 1, "b": 0})
except ZeroDivisionError as e:
    print("raised:", e)

raised: float division by zero


Inside an agent loop you usually want the model to *see* the failure and try something
else rather than crash the graph. Returning the problem as a string is the simplest
version of that:

In [9]:
@tool
def safe_divide(a: float, b: float) -> str:
    """Divide a by b. Returns an error message if b is zero."""
    if b == 0:
        return "Error: cannot divide by zero. Ask the user for a non-zero divisor."
    return str(a / b)


print(safe_divide.invoke({"a": 1, "b": 0}))
print(safe_divide.invoke({"a": 9, "b": 3}))

Error: cannot divide by zero. Ask the user for a non-zero divisor.
3.0


## Tools in a graph: `ToolNode`

In LangGraph "tool execution is handled by
[`ToolNode`](https://docs.langchain.com/oss/python/langchain/tools#tool-execution)".

`ToolNode` reads the **last message** in state, finds its `tool_calls`, runs the
matching tools, and appends a `ToolMessage` for each result.

Normally a model writes those `tool_calls`. But a tool call is just data, so we can
write one by hand:

In [10]:
manual_call = AIMessage(
    content="",
    tool_calls=[
        {"name": "add", "args": {"a": 1, "b": 2}, "id": "call_1", "type": "tool_call"}
    ],
)
manual_call.tool_calls

[{'name': 'add',
  'args': {'a': 1, 'b': 2},
  'id': 'call_1',
  'type': 'tool_call'}]

That dict — `name`, `args`, `id`, `type` — is the entire interface between a model and
a tool.

A state schema holding messages needs the `add_messages` reducer so each node appends
to the conversation instead of replacing it. Same mechanism as
[02-state.ipynb](./02-state.ipynb), with a purpose-built reducer.

In [11]:
class ToolState(TypedDict):
    messages: Annotated[list, add_messages]


toolkit = [add, subtract, multiply, calculate, round_to, safe_divide]

builder = StateGraph(ToolState)
builder.add_node("tools", ToolNode(toolkit))
builder.add_edge(START, "tools")
builder.add_edge("tools", END)

tool_graph = builder.compile()

result = tool_graph.invoke({"messages": [manual_call]})
for message in result["messages"]:
    print(f"{type(message).__name__:12} {message.content!r}")

AIMessage    ''
ToolMessage  '3.0'


The `ToolMessage` carrying `'3.0'` is the tool's return value on its way back to the
model.

> **Gotcha:** `ToolNode` must run **inside a compiled graph**. Calling
> `ToolNode([add]).invoke({"messages": [...]})` on its own raises
> `ValueError: Missing required config key 'N/A' for 'tools'`.

`ToolNode` also handles several calls at once, running them in parallel:

In [12]:
multi = AIMessage(
    content="",
    tool_calls=[
        {"name": "add", "args": {"a": 10, "b": 5}, "id": "c1", "type": "tool_call"},
        {"name": "calculator", "args": {"a": 10, "b": 5, "op": "mul"},
         "id": "c2", "type": "tool_call"},
        {"name": "safe_divide", "args": {"a": 10, "b": 0}, "id": "c3", "type": "tool_call"},
    ],
)

for message in tool_graph.invoke({"messages": [multi]})["messages"][1:]:
    print(f"{message.name:12} -> {message.content!r}")

add          -> '15.0'
calculator   -> '50.0'
safe_divide  -> 'Error: cannot divide by zero. Ask the user for a non-zero divisor.'
